# 第3章 コンデンサ評価（ノートブック版）

教科書の第3章の 3-6 の解析プログラム（`rc_fit.py`・`leak_analysis.py`・`lockin_analysis.py`）を、Pico で測る代わりに**合成データ**で、`run_ch3.sh` の解析の行の流れのとおりに1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約1分。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順・節 | 使うファイル |
|---|---|---|---|
| 1 | RC の充電・放電の合成データ（2% の遅い成分あり・なし）で、C の推定、2つの区間、偏り、残差の自己相関、ベイズの2段階目 | A の 4・5（3-3-1、図3-2） | `synth_rc.py`（**足したもの**）・`rc_fit.py` |
| 2 | 漏れ電流の2日分の合成データ（SQLite）で、ゼロ点、見かけの回帰、状態空間モデル4通りと AIC | B の 9（3-3-2、図3-4） | `synth_leak.py`（**足したもの**）・`leak_analysis.py` |
| 3 | デジタルロックインの計算を、理想の 100µF の波形でたどる | 3-3-3（図3-6） | （このノートブックの中の計算） |
| 4 | ロックインの合成データ（100µF・R_ref 1kΩ、0.1µF・R_ref 100kΩ と 10kΩ）で、Z(f)、等価回路の当てはめ、系統誤差の重み | C の 11・12 | `synth_lockin.py`（**足したもの**）・`lockin_analysis.py` |

- 次のセルたちが、3-6 の Mac 側の解析の3つのファイルと、合成データを作る3つのファイル（教科書にはない、ノートブックで足したもの）を、このノートブックの隣に `%%writefile` で書き出します（教科書の `setup_ch3.sh` で `~/home-lab/ch3/` に保存する代わり）。3-6 のファイルの中身は教科書と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- 各節では、`run_ch3.sh` の `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で教科書の手順の「確かめること」と 3-3・3-7 の値を確かめます。`ok()` は、PyMC のサンプリングの乱数によらずに成り立つことだけにしてあります。
- `rc_fit.py` のベイズの2段階目（PyMC）は、本文のコードのまま（1000点・調整 1500点。小さい設定の引数はない）で動かします。小さいモデルなので、1つのファイルで約10秒です。
- **入れていないもの（Colab ではできない手順）**：Pico A・Pico B での記録（A の 1〜3、B の 6〜8、C の 10・11 の `lockin_capture.py`）、Pi のロガーとスナップショットのコピー、テスタでの抵抗と ADC_VREF の測定。Pico 側のプログラム（`rc_step.py`・`leak_mqtt.py`（MicroPython）、`lockin.c`（C SDK））と Mac 側の記録のプログラム（`rc_capture.py`・`lockin_capture.py`）は、**読むためのセル**（実行しない）に載せてあります。測る代わりに、それぞれと同じ形の合成データを作ります。
- ⚠ 実物の測定の注意（電解コンデンサの向き、USB を抜いたときに節点L が負になること、放電させてから外すこと）は、教科書の 3-4・3-5 を読んでください。このノートブックは計算だけです。
- 数値は乱数の種を変えなければ教科書の図3-2・図3-4 と同じですが、scipy・statsmodels・PyMC の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels が最初から入っている。PyMC も入っていることが多い。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("pymc", "pymc"), ("statsmodels", "statsmodels"), ("pandas", "pandas"), ("scipy", "scipy")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import pymc, statsmodels, scipy, numpy
print("OK", "PyMC", pymc.__version__, "statsmodels", statsmodels.__version__, "scipy", scipy.__version__, "numpy", numpy.__version__)
print("（教科書は PyMC 6.4.0・statsmodels 0.15.0 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の6つのセルで、3-6 の `rc_fit.py`・`leak_analysis.py`・`lockin_analysis.py` と、合成データを作る `synth_rc.py`・`synth_leak.py`・`synth_lockin.py`（ノートブックで足したもの）を書き出します。

In [ ]:
%%writefile rc_fit.py
#!/usr/bin/env python3
"""第3章 A：RC の充電・放電から容量を推定する（非線形最小二乗 → ベイズ）

使い方: cd ~/home-lab/ch3 && uv run python rc_fit.py data_rc/C100u_*.csv --c-nom 100e-6 --r-rel 0.005
  --c-nom : 公称容量 [F]（比べるためだけに使う）
  --r-rel : 抵抗の実測値の相対誤差（テスタの確度。DE-200A の仕様で要確認。既定 0.5%）

1段目：1回の記録ごとに v(t) = v∞ + (v0 − v∞)·exp(−t/τ) を最小二乗で当てはめ、τ を求める
2段目：τ_k ~ Normal(τ_真, s)（回ごとのばらつき）、R ~ Normal(R_実測, R_実測·r_rel)、C = τ_真 / R
       を PyMC で推定し、C の 95% 信用区間を出す（回ごとの違いと R の誤差の両方を含む）
       充電と放電は同じ τ_真 のまわりのばらつきとしてまとめる（向きによる差は別に表示する）
偏りの目安：当てはめる範囲を最初の 3τ に縮めて当てはめ直し、τ が動くかを見る（3-3-1）
"""
import argparse
import json
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from statsmodels.tsa.stattools import acf

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=FutureWarning)


def model(t, vinf, v0, tau):
    return vinf + (v0 - vinf) * np.exp(-t / tau)


def fit_one(g, tau0):
    t, v = g["t_s"].to_numpy(), g["v"].to_numpy()
    p0 = [v[-1], v[0], tau0]
    popt, pcov = curve_fit(model, t, v, p0=p0, maxfev=20000)
    res = v - model(t, *popt)
    return popt, np.sqrt(np.diag(pcov)), res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv", nargs="+")
    ap.add_argument("--c-nom", type=float, default=None)
    ap.add_argument("--r-rel", type=float, default=0.005)
    ap.add_argument("--no-bayes", action="store_true")
    a = ap.parse_args()

    for path in a.csv:
        path = pathlib.Path(path)
        d = pd.read_csv(path)
        meta = json.loads(path.with_suffix(".json").read_text(encoding="utf-8"))
        R = meta["R"]
        print(f"\n===== {path.name}（R = {R:.0f} Ω）=====")
        rows = []
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
        for (rep, dirn), g in d.groupby(["rep", "dir"]):
            popt, se, res = fit_one(g, meta["tau_guess"])
            vinf, v0, tau = popt
            r1 = acf(res, nlags=1)[1]
            g3 = g[g["t_s"] <= g["t_s"].iloc[0] + 3 * tau]           # 最初の 3τ だけ
            tau3 = fit_one(g3, tau)[0][2] if len(g3) >= 20 else np.nan
            rows.append(dict(rep=rep, dir=dirn, tau=tau, tau_se=se[2], vinf=vinf, v0=v0,
                             res_sd=res.std(ddof=3), res_acf1=r1, tau3=tau3))
            print(f"rep {rep} {dirn:9s}: τ = {tau:.5g} s（1回の当てはめの標準誤差 {se[2]:.2g} s）、"
                  f"C = τ/R = {tau/R*1e6:.4g} µF、v∞ = {vinf:.4f} V、残差の標準偏差 {res.std()*1e3:.2f} mV、"
                  f"残差の自己相関 ρ(1) = {r1:.2f}")
            if rep == 0:
                ax[0].plot(g["t_s"], g["v"], ".", ms=1.5, label=f"{dirn}")
                ax[0].plot(g["t_s"], model(g["t_s"], *popt), "k-", lw=0.8)
                ax[1].plot(g["t_s"], res * 1e3, ".", ms=1.5, label=f"{dirn}")
        r = pd.DataFrame(rows)
        C_runs = r["tau"] / R
        print(f"\n回ごとの C：平均 {C_runs.mean()*1e6:.4g} µF、標準偏差 {C_runs.std(ddof=1)*1e6:.3g} µF（{len(r)} 回）")
        print(f"1回の当てはめの区間（±1.96·SE）の幅の平均：±{1.96*(r['tau_se']/R).mean()*1e6:.3g} µF")
        print(f"回ごとの値のばらつきから見た幅（±1.96·SD）：±{1.96*C_runs.std(ddof=1)*1e6:.3g} µF")
        print("・残差に自己相関があると、1回の当てはめの標準誤差は小さすぎる。温度などの回ごとに変わる誤差は、回ごとのばらつきに出る。")
        print("・どちらの幅も、毎回同じように出るモデルの合わなさ（誘電吸収）による偏りは含まない。")
        C3 = r["tau3"] / R
        print(f"当てはめる範囲を最初の 3τ に縮めたときの C：平均 {C3.mean()*1e6:.4g} µF"
              f"（全部で当てはめたときとの差 {(C3.mean() / C_runs.mean() - 1) * 100:+.2f}%。"
              "差が 0.1% 程度より大きければ、1本の指数関数では合っていない＝偏りがある。差は偏りの大きさの目安）")
        dd = r.groupby("dir")["tau"].mean() / R * 1e6
        if {"charge", "discharge"} <= set(dd.index):
            print(f"充電 {dd['charge']:.4g} µF、放電 {dd['discharge']:.4g} µF（差 {dd['charge']-dd['discharge']:+.3g} µF。"
                  "誘電吸収や漏れ電流があると差が出る）")
        if a.c_nom:
            print(f"公称値 {a.c_nom*1e6:.4g} µF との比 {C_runs.mean()/a.c_nom:.3f}（電解コンデンサの許容差は ±20% 程度のことが多い。データシートで要確認）")

        if not a.no_bayes and len(r) >= 2:
            import pymc as pm
            with pm.Model():
                tau_true = pm.LogNormal("tau_true", np.log(r["tau"].mean()), 1.0)
                s = pm.HalfNormal("s", r["tau"].mean() * 0.1)
                Rr = pm.Normal("R", R, R * a.r_rel)
                pm.Normal("tau_obs", tau_true, s, observed=r["tau"].to_numpy())
                C = pm.Deterministic("C", tau_true / Rr)
                idata = pm.sample(1000, tune=1500, chains=4, cores=1, target_accept=0.95,
                                  random_seed=3, progressbar=False)
            c = idata.posterior["C"].values.ravel() * 1e6
            div = int(idata.sample_stats["diverging"].values.sum())
            print(f"[ベイズ] C の事後平均 {c.mean():.4g} µF、95% 信用区間 {np.percentile(c, 2.5):.4g}〜"
                  f"{np.percentile(c, 97.5):.4g} µF（回ごとのばらつきと R の誤差 {a.r_rel*100:.1f}% を含む。発散 {div} 回）")
            print("  （充電と放電をまとめて1つの τ_真 としている。上の「充電・放電の差」が大きいときは、"
                  "向きによる系統的な差があるので、この区間をそのまま信じない）")

        ax[0].set(xlabel="時間 [s]", ylabel="コンデンサの電圧 [V]", title=f"{meta['cap']}（rep 0）")
        ax[1].axhline(0, color="k", lw=0.8)
        ax[1].set(xlabel="時間 [s]", ylabel="残差 [mV]", title="残差（波打つなら、モデルが合っていない部分がある）")
        for x in ax:
            x.grid(True, alpha=0.3)
            x.legend()
        fig.tight_layout()
        fig.savefig(path.with_suffix(".png"), dpi=130)
        plt.close(fig)
        print(f"{path.with_suffix('.png')} を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile leak_analysis.py
#!/usr/bin/env python3
"""第3章 B：漏れ電流の長時間記録（Pi の SQLite のスナップショット）を解析する

使い方: cd ~/home-lab/ch3 && uv run python leak_analysis.py lab_snapshot.db --dev pico1-leak --skip-h 2
  --dev    : leak_mqtt.py の DEV（第0章の DEVICE_ID に "-leak" を付けたもの）
  --skip-h : 最初の何時間を捨てるか（充電の直後は電流が大きく変わるので）
  --step   : 何分ごとの平均にまとめるか（既定 10 分）

やること
  1. JSON から漏れ電流 i_na（ゼロ点 V_Z を引いたもの）と室温 temp_c を取り出し、--step 分ごとの平均にする。
     ゼロ点 V_Z の大きさと温度による変化も表示する。平均が 0 以下の点は log がとれないので捨てる（その数も表示）
  2. 【誤りの例】log(電流) を温度だけで回帰する（時間とともに減る傾向と、温度の変化が混ざる）
  3. 状態空間モデル：log I_t = μ_t + β·(T_t − 平均) + ε_t、μ_t はゆっくり動く水準
     → 温度の影響 β（1℃ あたり何%変わるか）を、ゆっくりした減少と分けて推定する
     μ_t の動き方（ランダムウォーク・なめらかな傾向など）を4通り試し、AIC で比べる
     （AIC をそろえて比べるため、どのモデルも最初の2点を尤度の計算から外す）
"""
import argparse
import json
import sqlite3
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore")


def load(path, dev):
    con = sqlite3.connect(path)
    df = pd.read_sql_query("SELECT recv_unix, payload FROM messages WHERE dev = ? ORDER BY id", con, params=(dev,))
    if df.empty:
        sys.exit(f"dev = {dev} のデータがありません（--dev を確認。第0章の read_db.py でも確かめられる）")
    p = df["payload"].apply(json.loads)
    out = pd.DataFrame({"t": pd.to_datetime(df["recv_unix"], unit="s", utc=True).dt.tz_convert("Asia/Tokyo"),
                        "i_na": p.apply(lambda d: d.get("i_na")), "temp_c": p.apply(lambda d: d.get("temp_c")),
                        "v_z": p.apply(lambda d: d.get("v_z", np.nan))})
    return out.dropna(subset=["i_na", "temp_c"])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("db")
    ap.add_argument("--dev", required=True)
    ap.add_argument("--skip-h", type=float, default=2.0)
    ap.add_argument("--step", type=int, default=10)
    a = ap.parse_args()

    raw = load(a.db, a.dev)
    t0 = raw["t"].iloc[0]
    raw = raw[raw["t"] >= t0 + pd.Timedelta(hours=a.skip_h)]
    d = raw.set_index("t").resample(f"{a.step}min").mean().dropna(subset=["i_na", "temp_c"])
    if d["v_z"].notna().any():
        vz = d["v_z"].to_numpy() * 1e3
        ok = np.isfinite(vz)
        k = np.polyfit(d["temp_c"].to_numpy()[ok], vz[ok], 1)[0]
        print(f"ゼロ点 V_Z（AGND につないだ GP27 の読み）：平均 {vz[ok].mean():.3f} mV、"
              f"{vz[ok].min():.3f}〜{vz[ok].max():.3f} mV、温度とともに {k*1e3:+.1f} µV/°C（i_na からは引いてある）")
        if vz[ok].mean() < 0.05:
            print("  V_Z がほぼ 0：ゼロ点が負にずれていて 0 に張り付いている可能性がある（read_u16 は負にならない）")
    else:
        print("v_z がありません（古い leak_mqtt.py のデータ）。ゼロ点のずれは引かれていません")
    nbad = int((d["i_na"] <= 0).sum())
    if nbad:
        print(f"注意：{a.step} 分平均の電流が 0 以下の点が {nbad} 個あり、log がとれないので捨てます。"
              "雑音の床に近い小さな値だけを捨てるので、残りの平均は大きいほうに偏る")
    d = d[d["i_na"] > 0]
    if len(d) < 20:
        sys.exit("データが少なすぎます（--step を小さくするか、もっと長く記録する）")
    hours = (d.index[-1] - d.index[0]).total_seconds() / 3600
    print(f"{len(d)} 点（{a.step} 分平均）、{hours:.1f} 時間。電流 {d['i_na'].min():.1f}〜{d['i_na'].max():.1f} nA、"
          f"室温 {d['temp_c'].min():.1f}〜{d['temp_c'].max():.1f} °C")
    y = np.log(d["i_na"].to_numpy())
    T = d["temp_c"].to_numpy() - d["temp_c"].mean()

    # ---- 誤りの例：温度だけで回帰 ----
    ols = sm.OLS(y, sm.add_constant(T)).fit()
    b, (lo, hi) = ols.params[1], ols.conf_int()[1]
    print(f"\n[誤りの例] log I = a + β·T（温度だけ）：β = {b*100:+.2f} %/°C（95%区間 {lo*100:+.2f}〜{hi*100:+.2f}）、"
          f"ダービン・ワトソン比 {sm.stats.durbin_watson(ols.resid):.2f}")
    print("  電流がゆっくり減る傾向と温度の変化が同じ時間に起きると、その減少まで温度のせいにしてしまう（見かけの回帰）。")

    # ---- 状態空間モデル：水準の動き方を変えた4つを AIC で比べる ----
    print("\n[状態空間モデル] log I_t = μ_t + β·T_t + ε_t（μ_t の動き方を変えて比べる）")
    # どれも観測雑音 ε_t（irregular）を含む形で指定する
    specs = {"ランダムウォーク": dict(level="local level"),
             "ランダムウォーク＋一定の傾き": dict(level=True, stochastic_level=True, trend=True,
                                                 stochastic_trend=False, irregular=True),
             "なめらかな傾向（傾きがゆっくり変わる）": dict(level="smooth trend"),
             "ローカル線形トレンド": dict(level="local linear trend")}
    best = None
    for jp, kw in specs.items():
        m = sm.tsa.UnobservedComponents(y, exog=T, **kw)
        m.loglikelihood_burn = 2        # 水準が1つのモデルと2つのモデルで、尤度に使う点の数をそろえる
        r = m.fit(disp=False)
        bi = list(r.param_names).index("beta.x1")
        lo2, hi2 = r.conf_int()[bi]
        print(f"  {jp:22s} β = {r.params[bi]*100:+.2f} %/°C（95%区間 {lo2*100:+.2f}〜{hi2*100:+.2f}）、AIC = {r.aic:.1f}")
        if best is None or r.aic < best[0].aic:
            best = (r, jp, bi)
    r, jp, bi = best
    b2 = r.params[bi]
    print(f"  → AIC が最小：{jp}。β = {b2*100:+.2f} %/°C")
    print("  水準が自由に動くモデルほど、ゆっくりした温度の変化まで水準に吸い取られて β が小さく出やすい。")
    print("  β の区間が 0 をまたぐなら、この記録の温度の範囲では温度の影響ははっきりしない。")
    level = r.smoothed_state[0]

    fig, ax = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
    ax[0].semilogy(d.index, d["i_na"], ".", ms=3, label="測定（平均）")
    ax[0].semilogy(d.index, np.exp(level + b2 * T), "-", label="状態空間モデル（水準＋温度）")
    ax[0].semilogy(d.index, np.exp(level), "--", label="水準 μ_t だけ（温度の影響を除いた傾向）")
    ax[0].set_ylabel("漏れ電流 [nA]")
    ax[0].legend(fontsize=8)
    ax[1].plot(d.index, d["temp_c"], color="tab:red")
    ax[1].set_ylabel("室温 [°C]")
    ax[2].plot(d.index, y - level, ".", ms=3, label="log I − 水準")
    ax[2].plot(d.index, b2 * T, "-", label=f"β·(T − 平均)、β = {b2*100:+.1f} %/°C")
    ax[2].set_ylabel("log の差")
    ax[2].legend(fontsize=8)
    for x in ax:
        x.grid(True, which="both", alpha=0.3)
    fig.tight_layout()
    fig.savefig("leak.png", dpi=130)
    print("\nleak.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lockin_analysis.py
#!/usr/bin/env python3
"""第3章 C：デジタルロックインでインピーダンス Z(f) を求め、等価回路を当てはめる

使い方: cd ~/home-lab/ch3 && uv run python lockin_analysis.py data_lockin/C100u_1.npz --rref-rel 0.005
  --rref-rel : R_ref の実測値の相対誤差（テスタの確度。DE-200A の仕様で要確認）。C は R_ref に反比例するので C の区間に足す
  --fmin     : これより低い周波数 [Hz] の点を当てはめから外す（表には出す）。既定 0（全部使う）

やり方
  1. 周期ごとに、駆動側 v1 とコンデンサ側 v2 の基本波の複素振幅（フェーザ）を、sin・cos を掛けて平均して求める
     （ADC1 は ADC0 より 1/fs だけ後に変換されるので、その分の位相を補正する）
  2. Z = R_ref · V2 / (V1 − V2)。周期ごとの Z の平均と標準誤差（周期どうしのばらつきから）を出す
  3. 駆動側は方形波なので、等間隔の標本化ではエッジの位置が最大で半サンプルずれ、V1 の位相に
     ±π/N（N は1周期の点数）程度の系統誤差 δ が残る。Z = R_ref·V2/(V1 − V2) に入ると
     ΔZ ≈ −jδ·Z·(R_ref + Z)/R_ref になるので、δ を一様分布とみなし、|Z|·|R_ref + Z|/R_ref·(π/N)/√3 を
     Re Z・Im Z の誤差に足す（周期どうしのばらつきには現れない誤差。|Z| が R_ref より大きいと急に大きくなる）
  4. Z = R_s + 1/(jωC) を、3 の誤差で重みをつけた最小二乗で当てはめる（R_s は直列抵抗 ESR の目安）
"""
import argparse

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy import stats

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def impedance(npz):
    z = np.load(npz)
    rref = float(z["rref"])
    out = []
    for i, f in enumerate(z["f"]):
        n, p, fs = int(z["n"][i]), int(z["p"][i]), float(z["fs"][i])
        raw = z[f"raw{i}"].astype(float)
        v1, v2 = raw[0::2], raw[1::2]                  # ADC0 = 駆動側、ADC1 = コンデンサ側
        m = np.arange(len(v1))
        w = 2 * np.pi * f
        t1 = 2 * m / fs                                # ADC0 の時刻
        t2 = t1 + 1 / fs                               # ADC1 は 1サンプル分あと
        zs = []
        for k in range(p):
            s = slice(k * n, (k + 1) * n)
            V1 = 2 * np.mean(v1[s] * np.exp(-1j * w * t1[s]))
            V2 = 2 * np.mean(v2[s] * np.exp(-1j * w * t2[s]))
            zs.append(rref * V2 / (V1 - V2))
        zs = np.array(zs)
        zm = zs.mean()
        se = (zs.real.std(ddof=1) + 1j * zs.imag.std(ddof=1)) / np.sqrt(p)
        amp1 = np.abs(2 * np.mean(v1 * np.exp(-1j * w * t1)))
        amp2 = np.abs(2 * np.mean(v2 * np.exp(-1j * w * t2)))
        out.append(dict(f=f, Z=zm, se=se, p=p, n=n, amp1=amp1, amp2=amp2))
    return out, str(z["dut"]), rref


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("npz")
    ap.add_argument("--rref-rel", type=float, default=0.005)
    ap.add_argument("--fmin", type=float, default=0.0)
    a = ap.parse_args()

    res, dut, rref = impedance(a.npz)
    f = np.array([r["f"] for r in res])
    Z = np.array([r["Z"] for r in res])
    se = np.array([r["se"] for r in res])
    print(f"\n===== {dut}（R_ref = {rref:.0f} Ω）=====")
    nn = np.array([r["n"] for r in res])
    # 方形波の標本化による位相の系統誤差（目安）。|Z| ≫ R_ref では |Z|²/R_ref·δ になり、主に Im Z（C）に出る
    sys_err = np.abs(Z) * np.abs(rref + Z) / rref * (np.pi / nn) / np.sqrt(3)
    sre = np.sqrt(se.real ** 2 + sys_err ** 2)
    sim = np.sqrt(se.imag ** 2 + sys_err ** 2)
    # 図の誤差棒（95%）：周期のばらつきの部分は周期の数が少ない（1Hz で4周期）ので t 分布の値を掛ける
    tp = stats.t.ppf(0.975, np.array([r["p"] for r in res]) - 1)
    ere = np.sqrt((tp * se.real) ** 2 + (1.96 * sys_err) ** 2)
    eim = np.sqrt((tp * se.imag) ** 2 + (1.96 * sys_err) ** 2)
    use = f >= a.fmin
    print("   f [Hz]   Re Z [Ω] ± SE（周期）± 系統   −Im Z [Ω] ± SE（周期）   tanδ    |Z|/R_ref  |V2| [LSB]  周期数")
    for r, e, u in zip(res, sys_err, use):
        z = r["Z"]
        print(f"  {r['f']:7.1f}  {z.real:9.3f} ± {r['se'].real:6.3f} ± {e:6.3g}   {-z.imag:10.3f} ± {r['se'].imag:7.3f}"
              f"   {z.real / -z.imag:6.3f}   {abs(z) / rref:8.3f}   {r['amp2']:9.2f}   {r['p']}"
              + ("" if u else "   （当てはめに使わない）"))
    print("  tanδ = Re Z / (−Im Z)。電解コンデンサでは、低い周波数の Re Z に誘電体の損失（tanδ·|Z|）が入り、")
    print("  R_s（ESR）だけでは説明できない分になる。|Z|/R_ref が 1 より大きい点は、系統誤差が急に大きくなる。")
    if use.sum() < 2:
        raise SystemExit("--fmin が大きすぎて、当てはめに使う点が 2 つ未満です")
    f, Z, sre, sim, ere, eim = f[use], Z[use], sre[use], sim[use], ere[use], eim[use]

    w = 2 * np.pi * f

    def resid(th):
        rs, logc = th
        zm = rs + 1 / (1j * w * np.exp(logc))
        return np.concatenate([(zm.real - Z.real) / sre, (zm.imag - Z.imag) / sim])

    c0 = np.median(-1 / (w * Z.imag))
    fit = least_squares(resid, [0.5, np.log(abs(c0))])
    J = fit.jac
    dof = 2 * len(f) - 2
    s2 = np.sum(fit.fun ** 2) / dof
    try:
        cov = np.linalg.inv(J.T @ J) * max(s2, 1.0)    # 当てはまりが悪いときは区間を広げる
    except np.linalg.LinAlgError:
        raise SystemExit("当てはめられませんでした（Im Z がほぼ 0：コンデンサではなく抵抗を測ったのでは？）")
    rs, logc = fit.x
    se_rs, se_logc = np.sqrt(np.diag(cov))
    tq = stats.t.ppf(0.975, dof)
    C = np.exp(logc)
    print(f"\n等価回路 Z = R_s + 1/(jωC) の当てはめ（自由度 {dof}、規格化した残差の2乗和/自由度 = {s2:.2f}）")
    se_c = np.sqrt(se_logc ** 2 + a.rref_rel ** 2)    # log C の誤差に R_ref の相対誤差を足す
    print(f"  C   = {C*1e6:.4g} µF（95%区間 {np.exp(logc - tq*se_c)*1e6:.4g}〜{np.exp(logc + tq*se_c)*1e6:.4g}。"
          f"R_ref の誤差 {a.rref_rel*100:.1f}% を含む）")
    print(f"  R_s = {rs:.3f} Ω（95%区間 {rs - tq*se_rs:.3f}〜{rs + tq*se_rs:.3f}）")
    if rs - tq * se_rs < 0 < rs + tq * se_rs:
        print("  R_s の区間が 0 をまたぐ → このデータでは、直列抵抗（ESR）は決められない")
    if s2 > 3:
        print("  規格化した残差の2乗和/自由度が大きい → R_s + C のモデルが合っていない（誘電体の損失・ADC の非直線性など）")
    q = 3.3 / 4096
    print(f"\n[誤差の見積もり] ADC の1目盛り = {q*1e3:.2f} mV。高い周波数では |V2| が数〜数十目盛りしかない。")
    print("  R_s を見分けるには、Re Z の誤差（周期のばらつき＋系統誤差）が R_s より十分小さい必要がある。")
    print("  低い周波数では |Z| が大きいので、わずかな位相の誤差でも Re Z が大きくずれる（表の「系統」の列）。")
    print("  |Z| が R_ref より大きい点では、ずれが |Z|²/R_ref·δ と急に大きくなり、Im Z（つまり C）にも出る。")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    ax[0].errorbar(Z.real, -Z.imag, xerr=ere, yerr=eim, fmt="o", ms=4, label="測定（95%、系統誤差を含む）")
    ww = 2 * np.pi * np.logspace(np.log10(f.min()), np.log10(f.max()), 200)
    zf = rs + 1 / (1j * ww * C)
    ax[0].plot(zf.real, -zf.imag, "-", label="当てはめ")
    ax[0].set(xlabel="Re Z [Ω]", ylabel="−Im Z [Ω]", title=f"ナイキスト線図（{dut}）", yscale="log")
    ax[1].loglog(f, np.abs(Z), "o", label="|Z| 測定")
    ax[1].loglog(ww / 2 / np.pi, np.abs(zf), "-", label="当てはめ")
    ax[1].loglog(f, 1 / (2 * np.pi * f * C), ":", color="gray", label="1/(ωC)")
    ax[1].set(xlabel="周波数 [Hz]", ylabel="|Z| [Ω]", title="ボード線図（大きさ）")
    for x in ax:
        x.grid(True, which="both", alpha=0.3)
        x.legend(fontsize=8)
    fig.tight_layout()
    png = a.npz.replace(".npz", ".png")
    fig.savefig(png, dpi=130)
    print(f"{png} を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile synth_rc.py
#!/usr/bin/env python3
"""ノートブック版で足したもの：rc_capture.py と同じ形の CSV・JSON を、RC の式から作る

使い方: python synth_rc.py --slow 0.02 --name C100u_x     （2% の遅い成分あり。図3-2 の合成データ）
        python synth_rc.py --slow 0    --name C100u_clean （遅い成分なし）
  v(t) = v∞ + (v0 − v∞)·[(1 − a)·exp(−t/τ) + a·exp(−t/(8τ))]、τ = RC（R = 9980Ω、C = 103µF の真の値）
  a（--slow）は誘電吸収のつもりの遅い成分の割合で、毎回同じに入れる。雑音は 0.6mV、12ビットの ADC に丸める。
  記録の長さと間隔は rc_capture.py と同じ決め方（7τ を 3000 点まで、間隔は 100µs 以上）。充電・放電を3回ずつ
"""
import argparse
import json
import pathlib

import numpy as np
import pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument("--slow", type=float, default=0.02)
ap.add_argument("--name", default="C100u_x")
a = ap.parse_args()

rng = np.random.default_rng(1)
R = 9980
C = 103e-6
vref = 3.296
tau = R * C
tau_guess = R * 100e-6                               # rc_capture.py の --tau（R × 公称容量）
dur = 7 * tau_guess
dt_us = max(100, int(dur / 3000 * 1e6))
n = min(3000, int(dur * 1e6 / dt_us))
rows = []
for rep in range(3):
    for dirn in ("charge", "discharge"):
        t = np.arange(n) * dt_us / 1e6 + 30e-6
        a2 = a.slow
        if dirn == "charge":
            v = 3.29 * (1 - (1 - a2) * np.exp(-t / tau) - a2 * np.exp(-t / (8 * tau)))
        else:
            v = 3.29 * ((1 - a2) * np.exp(-t / tau) + a2 * np.exp(-t / (8 * tau)))
        v += rng.normal(0, 0.6e-3, n)
        raw = np.clip(np.round(v / vref * 4096), 0, 4095) * 16   # 12ビットの値を read_u16() と同じ 16ビットの目盛りに
        for ti, ri in zip(t, raw):
            rows.append(dict(cap="C100u", rep=rep, dir=dirn, t_s=round(ti, 6), v=ri / 65536 * vref))
p = pathlib.Path("data_rc")
p.mkdir(exist_ok=True)
pd.DataFrame(rows).to_csv(p / f"{a.name}.csv", index=False)
json.dump(dict(cap="C100u", R=R, tau_guess=tau_guess, vref=vref, dt_us=dt_us, n=n, reps=3), open(p / f"{a.name}.json", "w"))
print(f"data_rc/{a.name}.csv と .json を作りました（真の τ = {tau:.5f} s、C = {C*1e6:.1f} µF、遅い成分 {a2*100:.0f}%、"
      f"1回 {n} 点、{dt_us} µs ごと）")

In [ ]:
%%writefile synth_leak.py
#!/usr/bin/env python3
"""ノートブック版で足したもの：Pi の lab.db のスナップショットと同じ形の SQLite を、漏れ電流の式から作る

使い方: python synth_leak.py --out lab_snapshot.db     （2日分、10秒ごと、dev = pico1-leak）
  I(t) = 3nA·(1 + 30·exp(−t/3h))·(1 + 0.3·exp(−t/30h))·exp(β·(T − 25))、真の β = +4%/℃
  室温 T は 25 ± 3℃ の1日周期。ADC のゼロ点は 3mV で、温度とともに 80µV/℃ 変わる（GP27 の読みとして引く）
  R_S = 1MΩ、ADC は 256回平均（1回ごとに 1.5目盛りの雑音）。i_na = (v_l − v_z)/R_S は leak_mqtt.py と同じ計算
"""
import argparse
import json
import os
import sqlite3

import numpy as np

ap = argparse.ArgumentParser()
ap.add_argument("--out", default="lab_snapshot.db")
ap.add_argument("--rs", type=float, default=1e6)
a = ap.parse_args()

R_S = a.rs
TC = 0.08e-3
rng = np.random.default_rng(2)
if os.path.exists(a.out):
    os.remove(a.out)
con = sqlite3.connect(a.out)
con.execute("CREATE TABLE messages(id INTEGER PRIMARY KEY AUTOINCREMENT, recv_unix REAL, topic TEXT, payload TEXT, dev TEXT, seq INTEGER)")
t0 = 1.79e9
n = int(48 * 3600 / 10)
beta = 0.04
VREF = 3.3
NAVG = 256
LSB = VREF / 4096
for k in range(n):
    th = k * 10 / 3600
    T = 25 + 3 * np.sin(2 * np.pi * (th - 9) / 24) + 0.05 * rng.standard_normal()
    I = 3e-9 * (1 + 30 * np.exp(-th / 3)) * (1 + 0.3 * np.exp(-th / 30)) * np.exp(beta * (T - 25))
    off = 3e-3 + TC * (T - 25)              # ADC のゼロ点のずれ [V]（温度で変わる）

    def adc(v):
        r = np.clip(np.round((v + off) / LSB + rng.normal(0, 1.5, NAVG)), 0, 4095) * 16
        return r.mean() * VREF / 65536

    vl = adc(I * R_S)
    vz = adc(0.0)
    p = dict(dev="pico1-leak", seq=k, v_l=round(vl, 6), v_z=round(vz, 6), i_na=round((vl - vz) / R_S * 1e9, 3), temp_c=round(T, 2))
    con.execute("INSERT INTO messages(recv_unix,topic,payload,dev,seq) VALUES(?,?,?,?,?)",
                (t0 + k * 10, "lab/pico1-leak/telemetry", json.dumps(p), "pico1-leak", k))
con.commit()
print(f"{a.out} を作りました（{n} 行、48時間、真の β = +{beta*100:.0f}%/℃）")

In [ ]:
%%writefile synth_lockin.py
#!/usr/bin/env python3
"""ノートブック版で足したもの：lockin_capture.py と同じ形の npz を、RC の回路の式から作る

使い方: python synth_lockin.py C Rs Rref out.npz     （例：python synth_lockin.py 100e-6 0.3 998 data_lockin/C100u_1.npz）
  駆動 v1 は 0〜3.3V の方形波、v2 = 方形波のフーリエ級数の各高調波に H = Z/(R_ref + Z)、Z = R_s + 1/(jωC) を掛けた定常応答。
  lockin.c と同じ10個の周波数・1周期の点数 n・周期の数 p。ADC0（v1）と ADC1（v2）は交互で、ADC1 は 1/fs あと。
  開始の位相はでたらめ（方形波のエッジの位置が標本の間のどこにあるか分からない）。雑音 0.6mV、12ビットに丸める
"""
import sys

import numpy as np

C = float(sys.argv[1])
Rs = float(sys.argv[2])
rref = float(sys.argv[3])
out = sys.argv[4]
rng = np.random.default_rng(5)
FREQS = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
d = {}
fl, nl, pl, fsl = [], [], [], []
for i, f in enumerate(FREQS):
    n = 200
    while 2 * n * f < 800:
        n *= 2
    p = 4 if f < 2 else 8 if f < 10 else 20
    if 2 * n * p > 40000:
        p = 40000 // (2 * n)
    fs = 2 * n * f
    count = 2 * n * p
    k = np.arange(count)
    t = k / fs + rng.uniform(0, 1 / f)       # 開始の位相はでたらめ
    t1 = t[0::2]
    t2 = t[1::2]
    v1 = np.where((t1 * f) % 1.0 < 0.5, 3.3, 0.0)
    # 方形波 = 1.65 + Σ_{奇数 h} (2·3.3/(πh))·sin(2πhft) の各成分に H を掛ける
    v2 = np.full_like(t2, 1.65)
    for h in range(1, 4000, 2):
        w = 2 * np.pi * h * f
        Z = Rs + 1 / (1j * w * C)
        H = Z / (rref + Z)
        v2 += 2 * 3.3 / (np.pi * h) * np.imag(H * np.exp(1j * w * t2))
    q = 3.3 / 4096
    r1 = np.clip(np.round((v1 + rng.normal(0, 0.6e-3, len(v1))) / q), 0, 4095)
    r2 = np.clip(np.round((v2 + rng.normal(0, 0.6e-3, len(v2))) / q), 0, 4095)
    raw = np.empty(count, dtype=np.uint16)
    raw[0::2] = r1
    raw[1::2] = r2
    d[f"raw{i}"] = raw
    fl.append(f); nl.append(n); pl.append(p); fsl.append(fs)
np.savez(out, dut=f"C{C}", rref=rref, f=np.array(fl, float), n=np.array(nl), p=np.array(pl), fs=np.array(fsl, float), **d)
print(f"{out} を作りました（C = {C*1e6:.4g} µF、R_s = {Rs} Ω、R_ref = {rref:.0f} Ω）")

In [ ]:
import glob, os, re, shlex, subprocess, sys
from IPython.display import Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):   # data_rc/C100u_*.csv のような * を、ターミナルと同じく名前の順に広げる
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines()
                    if "findfont" not in l and not re.match(r"(Initializing NUTS|Sequential sampling|NUTS:|Sampling \d)", l))
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=760):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def near(got, want, d):
    """got が want と ±d 以内か（d は表示の最後の桁の 1 つ分。numpy・scipy の版で最後の桁が変わる分の幅）。リストなら要素ごと（d もリストにできる）"""
    if isinstance(want, (list, tuple)):
        got = list(got)
        ds = d if isinstance(d, (list, tuple)) else [d] * len(want)
        return len(got) == len(want) and all(near(g, w, e) for g, w, e in zip(got, want, ds))
    return abs(float(got) - want) <= d * (1 + 1e-6)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i))


os.makedirs("data_rc", exist_ok=True)
os.makedirs("data_lockin", exist_ok=True)
os.environ.setdefault("PYTENSOR_FLAGS", "compiledir=" + os.path.abspath("pytensor_cache"))   # PyMC のコンパイルの置き場所（このフォルダの中）
print("準備ができました")

## 1. RC の充電・放電から容量を求める（3-3-1、手順 A の 4・5）

### Pico での記録（読むだけ。このノートブックでは実行しません）

教科書では、Pico A に `rc_step.py` を `main.py` として入れ、Mac の `rc_capture.py` で充電・放電を3回ずつ記録します（`run_ch3.sh` の「A」）。`rc_capture.py` は `data_rc/<cap>_<日時>.csv`（列 `cap, rep, dir, t_s, v`）と `.json`（`R`・`tau_guess`・`vref` など）を作ります。

**Pico A（MicroPython）｜~/home-lab/ch3/pico/rc_step.py（Pico には main.py として保存）**

```python
# 第3章 A：RC の充電・放電を記録する（Pico A / MicroPython）。main.py として保存し、Mac の rc_capture.py から操作する
# 書き込み: mpremote connect auto fs cp pico/rc_step.py :main.py
#           mpremote connect auto reset
#
# 配線（図3-6 (A)）: GP15（ピン20）→ R（10kΩ または 100kΩ）→ 節点C → コンデンサの＋、コンデンサの− → AGND（ピン33）
#               節点C → GP26/ADC0（ピン31）
#
# シリアルコマンド（1行ずつ）
#   PING                  → PONG
#   LEVEL 0|1             → GP15 を Low / High にする（放電・充電の準備）→ OK
#   STEP 1|0 dt_us n      → GP15 を High（1、充電）か Low（0、放電）に切り替え、その瞬間から
#                           dt_us ごとに n 点、ADC0 を記録する。D,t_us,raw（0〜65535）の行を出して END
import sys
import time
from array import array
from machine import Pin, ADC, mem32

DRIVE_PIN = 15
ADC_GPIO = 26
NAVG = 4                        # 1点あたりの平均回数
NMAX = 6000

# ---- RP2350 エラッタ E9 対策：ADC ピンのデジタル入力バッファとプル抵抗を切る（第1章と同じ）----
PADS_BANK0_BASE = 0x40038000    # RP2350 データシート「PADS_BANK0」で要確認
IE_BIT, PUE_BIT, PDE_BIT = 1 << 6, 1 << 3, 1 << 2


def disable_digital_input(gpio):
    a = PADS_BANK0_BASE + 4 + 4 * gpio
    v = mem32[a] & 0xFFFFFFFF
    if v & ~0x1FF:
        print("WARN pad register looks wrong: GP%d = 0x%08x（書き換えません）" % (gpio, v))
        return
    mem32[a] = v & ~(IE_BIT | PUE_BIT | PDE_BIT)


drive = Pin(DRIVE_PIN, Pin.OUT, value=0)
adc = ADC(Pin(ADC_GPIO))
disable_digital_input(ADC_GPIO)          # ADC を初期化した「あと」に切る
ts = array("i", [0] * NMAX)
vs = array("H", [0] * NMAX)


def read_avg():
    s = 0
    for _ in range(NAVG):
        s += adc.read_u16()
    return s // NAVG


def step(level, dt_us, n):
    n = min(n, NMAX)
    t0 = time.ticks_us()
    drive.value(level)                    # ここが t = 0
    for i in range(n):
        target = i * dt_us
        while time.ticks_diff(time.ticks_us(), t0) < target:
            pass
        ts[i] = time.ticks_diff(time.ticks_us(), t0)
        vs[i] = read_avg()
    for i in range(n):                    # 記録が終わってから出力する
        print("D,%d,%d" % (ts[i], vs[i]))
    print("END")


def handle(line):
    p = line.split()
    if not p:
        return
    c = p[0].upper()
    if c == "PING":
        print("PONG")
    elif c == "LEVEL":
        drive.value(int(p[1]))
        print("OK")
    elif c == "STEP":
        step(int(p[1]), int(p[2]), int(p[3]))
    else:
        print("ERR 分からないコマンド:", line)


print("READY rc_step")
while True:
    try:
        handle(sys.stdin.readline().strip())
    except Exception as e:
        drive.value(0)
        print("ERR", e)
```

**Mac｜~/home-lab/ch3/rc_capture.py**

```python
#!/usr/bin/env python3
"""第3章 A：Pico の rc_step.py を操作して、RC の充電・放電を記録する

使い方: cd ~/home-lab/ch3 && uv run python rc_capture.py --cap C100u --r 9980 --tau 1.0 --reps 3
  --cap : コンデンサの名前（ファイル名に使う。C100u, C470u, C1000u, C0u1 など）
  --r   : 直列抵抗 R の実測値 [Ω]
  --tau : 時定数の見積もり [s]（R×公称容量）。記録の長さと間隔をこれから決める
  --vref: ADC_VREF の実測値 [V]
保存先: data_rc/<cap>_<日時>.csv（列 cap, rep, dir, t_s, v）と .json（条件）
"""
import argparse
import csv
import datetime
import glob
import json
import pathlib
import sys
import time

import serial


def find_port():
    ports = sorted(glob.glob("/dev/cu.usbmodem*"))
    if not ports:
        sys.exit("Pico のシリアルポートが見つかりません")
    return ports[0]


class Pico:
    def __init__(self, port):
        self.s = serial.Serial(port, 115200, timeout=1)
        time.sleep(0.3)
        self.s.reset_input_buffer()

    def command(self, line, timeout=120):
        self.s.write((line + "\n").encode())
        out, t0 = [], time.time()
        while time.time() - t0 < timeout:
            ln = self.s.readline().decode(errors="ignore").strip()
            if not ln:
                continue
            if ln.startswith("ERR"):
                sys.exit(f"Pico がエラーを返しました: {ln}")
            out.append(ln)
            if ln in ("OK", "END", "PONG"):
                return out
        sys.exit(f"応答がありません（{line}）。main.py が rc_step.py か確認してください")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cap", required=True)
    ap.add_argument("--r", type=float, required=True)
    ap.add_argument("--tau", type=float, required=True)
    ap.add_argument("--vref", type=float, default=3.3)
    ap.add_argument("--reps", type=int, default=3)
    ap.add_argument("--npts", type=int, default=3000)
    ap.add_argument("--port", default=None)
    a = ap.parse_args()

    dur = 7 * a.tau                                  # 7τ（残り 0.1%）まで記録する
    dt_us = max(100, int(dur / a.npts * 1e6))        # MicroPython で間に合う最短 100µs
    n = min(a.npts, int(dur * 1e6 / dt_us))
    print(f"1回の記録：{n} 点、{dt_us} µs ごと（{n*dt_us/1e6:.1f} 秒）。1回の充電＋放電で約 {2*(dur+7*a.tau):.0f} 秒")
    p = Pico(a.port or find_port())
    p.command("PING")
    rows = []
    for rep in range(a.reps):
        for level, dirn in ((1, "charge"), (0, "discharge")):
            p.command(f"LEVEL {1 - level}")             # 逆の状態で十分に待ってから切り替える
            print(f"  rep {rep} {dirn}：{7*a.tau:.0f} 秒待ってから記録します")
            time.sleep(7 * a.tau)
            for ln in p.command(f"STEP {level} {dt_us} {n}", timeout=dur + 60):
                if ln.startswith("D,"):
                    _, t_us, raw = ln.split(",")
                    rows.append({"cap": a.cap, "rep": rep, "dir": dirn,
                                 "t_s": int(t_us) / 1e6, "v": int(raw) / 65536 * a.vref})
    p.command("LEVEL 0")
    out = pathlib.Path("data_rc")
    out.mkdir(exist_ok=True)
    base = out / f"{a.cap}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
    with open(base.with_suffix(".csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    base.with_suffix(".json").write_text(json.dumps(
        {"cap": a.cap, "R": a.r, "tau_guess": a.tau, "vref": a.vref, "dt_us": dt_us, "n": n, "reps": a.reps},
        ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"保存しました: {base}.csv / .json（{len(rows)} 行）")


if __name__ == "__main__":
    main()
```

### 合成データで解析する

`synth_rc.py` は `rc_capture.py` と同じ形のファイルを、真の値 $R = 9980\,\Omega$・$C = 103.0\,\mu$F（$\tau = 1.028$s）の式から作ります。`--slow 0.02` は誘電吸収のつもりの 2% の遅い成分（時定数 8τ）を毎回同じに入れたもの（図3-2 の合成データ）、`--slow 0` は遅い成分なしです。`rc_fit.py` の表示で、「1回の当てはめの区間」と「回ごとのばらつきから見た幅」、「最初の 3τ に縮めたときの C」、残差の自己相関 ρ(1) を見てください。

In [ ]:
run("synth_rc.py --slow 0.02 --name C100u_x")
run("synth_rc.py --slow 0 --name C100u_clean")
out_rc = run("rc_fit.py data_rc/C100u_x.csv --c-nom 100e-6")   # 教科書：rc_fit.py data_rc/C100u_*.csv --c-nom 100e-6
show("data_rc/C100u_x.png")

In [ ]:
out_rc0 = run("rc_fit.py data_rc/C100u_clean.csv --c-nom 100e-6")

In [ ]:
# 確かめ（手順 A の 4・5、3-3-1 と図3-2）
C = num(r"回ごとの C：平均 ([\d.]+) µF", out_rc)
w1 = num(r"1回の当てはめの区間（±1.96·SE）の幅の平均：±([\d.]+) µF", out_rc)
wr = num(r"回ごとの値のばらつきから見た幅（±1.96·SD）：±([\d.]+) µF", out_rc)
ok(near(C, 104.6, 0.1) and near(w1, 0.029, 0.001) and near(wr, 0.009, 0.001),
   f"遅い成分 2%：C = 104.6 µF、回ごとの幅 ±0.009 µF のほうが1回の当てはめの幅 ±0.029 µF より狭い（3-3-1）")
ok(C - max(w1, wr) > 103.0, "真の値 103.0 µF はどちらの区間にも入らない（+1.6% の偏りは区間から分からない）")
d3 = num(r"全部で当てはめたときとの差 ([-+\d.]+)%", out_rc)
ok(near(num(r"最初の 3τ に縮めたときの C：平均 ([\d.]+) µF", out_rc), 103.7, 0.1) and abs(d3) > 0.1,
   f"最初の 3τ だけで当てはめ直すと C が 103.7 µF に動く（{d3:+.2f}%。0.1% より大きい → 偏りがある）")
r1 = [float(x) for x in re.findall(r"ρ\(1\) = ([-\d.]+)", out_rc)]
ok(len(r1) == 6 and min(r1) >= 0.5 and near(r1[0], 0.94, 0.01), f"残差がゆっくり波打ち、自己相関 ρ(1) = 0.94（0.5 以上）")
ok(0.8 <= num(r"公称値 100 µF との比 ([\d.]+)", out_rc) <= 1.2, "公称値の ±20% に入る")
lo, hi = (num(r"95% 信用区間 ([\d.]+)〜([\d.]+) µF", out_rc, k) for k in (1, 2))
ok(lo < C < hi and (hi - lo) / 2 > wr, f"ベイズの信用区間 {lo}〜{hi} µF は C を含み、R の誤差 0.5% のぶん回ごとの幅より広い")
d30 = num(r"全部で当てはめたときとの差 ([-+\d.]+)%", out_rc0)
r10 = [float(x) for x in re.findall(r"ρ\(1\) = ([-\d.]+)", out_rc0)]
ok(near(num(r"回ごとの C：平均 ([\d.]+) µF", out_rc0), 103.0, 0.1) and abs(d30) < 0.1 and max(abs(x) for x in r10) < 0.1,
   f"遅い成分なし：C = 103.0 µF、3τ に縮めた差 {d30:+.2f}%（0.1% 未満）、ρ(1) ≈ 0（残差が雑音のよう）")

## 2. 漏れ電流と温度（3-3-2、手順 B の 9）

### Pico での記録（読むだけ。このノートブックでは実行しません）

教科書では、Pico A の `leak_mqtt.py` が10秒ごとに漏れ電流 `i_na`（ゼロ点 `v_z` を引いたもの）と室温 `temp_c` を MQTT で Pi に送り、Pi の lab-logger が SQLite に書きます。2日ほど記録したら、Pi でスナップショットを作って Mac にコピーし、`leak_analysis.py` で解析します（`run_ch3.sh` の「B」）。Wi-Fi と MQTT の設定は第0章の `labconfig.py` にあります（このノートブックには実物の値を書きません）。

**Pico A（MicroPython）｜~/home-lab/ch3/pico/leak_mqtt.py（Pico には main.py として保存）**

```python
# 第3章 B：コンデンサの漏れ電流と室温を、MQTT で Raspberry Pi に送り続ける（Pico A / MicroPython）
# 準備: 第0章の labconfig.py と umqtt.simple、入門3 の :lib/bme280.py
# 書き込み: mpremote connect auto fs cp pico/leak_mqtt.py :main.py
#           mpremote connect auto reset
#
# 配線（図3-6 (B)）: 3V3(OUT)（ピン36）→ コンデンサの＋、コンデンサの− = 節点L → R_S（1MΩ）→ AGND（ピン33）
#                   節点L → 保護の 10kΩ → GP26/ADC0（ピン31）、GP26 と AGND の間に 0.1µF
#                   GP27/ADC1（ピン32）→ AGND に直結（ゼロ点の測定用）
#                   BME280：入門3 と同じ（3V3・GND・SDA=GP4・SCL=GP5）
# 漏れ電流 I = (V_L − V_Z) / R_S（ローサイドで測る。V_Z は AGND につないだ GP27 の読み = ADC のゼロ点のずれ）
# 通電中の V_L は 0〜3.3V に収まるが、USB を抜いた瞬間は節点L が 0V より下がる。10kΩ は GP26 に流れる電流を抑えるためのもの
import time
import json
import network
import rp2
from machine import Pin, ADC, I2C, mem32
from umqtt.simple import MQTTClient
from bme280 import BME280
import labconfig as cfg

R_S = 1_000_000.0        # R_S の実測値 [Ω] に書き換える
VREF = 3.300             # ADC_VREF の実測値 [V] に書き換える
PERIOD_S = 10            # 送る間隔 [s]
NAVG = 256               # 1回の測定で ADC を何回平均するか（GP26 と GP27 を交互に読む）
DEV = cfg.DEVICE_ID + "-leak"
TOPIC = ("lab/%s/telemetry" % DEV).encode()
ADC_GPIO = 26            # 節点L（10kΩ を通して）
ZERO_GPIO = 27           # AGND に直結（ゼロ点）

PADS_BANK0_BASE = 0x40038000    # 第1章と同じ E9 対策（RP2350 データシートで要確認）


def disable_digital_input(gpio):
    a = PADS_BANK0_BASE + 4 + 4 * gpio
    v = mem32[a] & 0xFFFFFFFF
    if v & ~0x1FF:
        print("WARN pad register looks wrong: GP%d = 0x%08x" % (gpio, v))
        return
    mem32[a] = v & ~((1 << 6) | (1 << 3) | (1 << 2))


rp2.country("JP")
adc = ADC(Pin(ADC_GPIO))
adc_z = ADC(Pin(ZERO_GPIO))
disable_digital_input(ADC_GPIO)
disable_digital_input(ZERO_GPIO)
env = BME280(I2C(0, sda=Pin(4), scl=Pin(5), freq=100_000))
led = Pin("LED", Pin.OUT)


def read_vl_vz():
    # GP26（節点L）と GP27（AGND）を交互に読むので、ゼロ点のずれとそのゆっくりした変化（温度など）を引ける
    sl = 0
    sz = 0
    for _ in range(NAVG):
        sl += adc.read_u16()
        sz += adc_z.read_u16()
    k = VREF / 65536 / NAVG
    return sl * k, sz * k


def wifi_connect(timeout_s=30):
    wlan = network.WLAN(network.STA_IF)
    wlan.active(True)
    if not wlan.isconnected():
        wlan.connect(cfg.WIFI_SSID, cfg.WIFI_PASS)
        t0 = time.ticks_ms()
        while not wlan.isconnected():
            if time.ticks_diff(time.ticks_ms(), t0) > timeout_s * 1000:
                raise RuntimeError("Wi-Fi接続に失敗 status=%d" % wlan.status())
            time.sleep_ms(250)


def main():
    seq = 0
    client = None
    while True:
        t, p, h = env.read()
        vl, vz = read_vl_vz()
        msg = {"dev": DEV, "seq": seq, "ticks_ms": time.ticks_ms(), "v_l": round(vl, 6), "v_z": round(vz, 6),
               "i_na": round((vl - vz) / R_S * 1e9, 3), "temp_c": round(t, 2), "press_pa": round(p, 1)}
        seq += 1
        try:
            if client is None:
                wifi_connect()
                client = MQTTClient(("pico-" + DEV).encode(), cfg.MQTT_HOST, port=1883,
                                    user=cfg.MQTT_USER, password=cfg.MQTT_PASS, keepalive=60)
                client.connect()
            client.publish(TOPIC, json.dumps(msg).encode())
            led.toggle()
        except Exception as e:
            print("通信エラー:", e)
            client = None
        print(msg)
        time.sleep(PERIOD_S)


main()
```

### 合成データで解析する

`synth_leak.py` は、Pi のスナップショットと同じ形（表 `messages`、列 `recv_unix`・`payload`・`dev` など）の `lab_snapshot.db` を作ります。2日分・10秒ごと、真の β = +4%/℃、ADC のゼロ点 3mV とその温度変化 80µV/℃ を入れたもの（図3-4 の合成データ）です。`run_ch3.sh` の最後の行と同じ引数で解析します。

In [ ]:
run("synth_leak.py --out lab_snapshot.db")
out_lk = run("leak_analysis.py lab_snapshot.db --dev pico1-leak --skip-h 2")
show("leak.png")

In [ ]:
# 確かめ（手順 B の 9、3-3-2 と図3-4）
ok(abs(num(r"ゼロ点 V_Z.*?平均 ([\d.]+) mV", out_lk) - 3.0) < 0.05 and abs(num(r"温度とともに ([-+\d.]+) µV/°C", out_lk) - 80) < 5,
   "ゼロ点 V_Z は約3mV で、温度とともに約 +80µV/°C（合成データの真の値）。i_na からは引いてある")
b_ols = num(r"\[誤りの例\].*?β = ([-+\d.]+) %/°C", out_lk)
ok(abs(b_ols - (-9.33)) < 0.05 and num(r"ダービン・ワトソン比 ([\d.]+)", out_lk) < 0.1,
   f"温度だけで回帰すると β = {b_ols:+.2f}%/℃ と符号まで逆（真の値 +4）、ダービン・ワトソン比が0に近い")
sp = {m.group(1).strip(): (float(m.group(2)), float(m.group(3)), float(m.group(4)), float(m.group(5)))
      for m in re.finditer(r"^  (\S+)\s+β = ([-+\d.]+) %/°C（95%区間 ([-+\d.]+)〜([-+\d.]+)）、AIC = ([-\d.]+)", out_lk, re.M)}
ok(len(sp) == 4, "状態空間モデルを4通り当てはめた")
ok(abs(sp["ランダムウォーク"][0] - (-3.46)) < 0.05, "水準が自由に動くランダムウォークでは β = −3.46%/℃（小さく出る）")
best = min(sp, key=lambda k: sp[k][3])
ok(best.startswith("なめらかな傾向") and abs(sp[best][0] - 3.43) < 0.05 and abs(sp[best][1] - 0.85) < 0.05 and abs(sp[best][2] - 6.01) < 0.05,
   "AIC が最小のなめらかな傾向で β = +3.43%/℃（95%区間 +0.85〜+6.01）")
ok(sp[best][1] < 4 < sp[best][2], "AIC が最小のモデルの区間は、真の値 +4%/℃ を含む")

## 3. デジタルロックインの計算をたどる（3-3-3、図3-6）

図3-5 (a) の計算を、1Hz の方形波・$R_{\mathrm{ref}} = 1$kΩ・理想の 100µF（ESR 0）・1周期 400 点でたどります（この節は Pico のデータを使わない、ノートブックの中だけの計算です）。$v_2$ は RC の充電・放電の定常状態の式で作ります。

$$V = \frac{2}{N}\sum_{k=0}^{N-1} v_k\,e^{-j\omega t_k},\qquad Z = R_{\mathrm{ref}}\,\frac{V_2}{V_1 - V_2}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）

f, N, Rref, Cid = 1.0, 400, 1000.0, 100e-6
w, tau, Tp = 2 * np.pi * f, Rref * Cid, 1 / f
a = np.exp(-Tp / 2 / tau)
vlo = 3.3 * a * (1 - a) / (1 - a * a)                     # 定常状態で、立ち上がりの直前の v2
vhi = 3.3 + (vlo - 3.3) * a                               # 立ち下がりの直前の v2


def v1f(t):
    return np.where((t * f) % 1 < 0.5, 3.3, 0.0)


def v2f(t):
    ph = (t * f) % 1 * Tp
    return np.where(ph < Tp / 2, 3.3 + (vlo - 3.3) * np.exp(-ph / tau), vhi * np.exp(-(ph - Tp / 2) / tau))


tk = (np.arange(N) + 0.5) / (N * f)
mc, ms = np.mean(v2f(tk) * np.cos(w * tk)), np.mean(v2f(tk) * np.sin(w * tk))
V1 = 2 / N * np.sum(v1f(tk) * np.exp(-1j * w * tk))
V2 = 2 / N * np.sum(v2f(tk) * np.exp(-1j * w * tk))
Zid = Rref * V2 / (V1 - V2)
C_back = 1 / (w * abs(Zid.imag))
print(f"v2·cos ωt の1周期の平均 {mc:+.3f}、v2·sin ωt の平均 {ms:+.3f}、v2 の平均（直流）{v2f(tk).mean():.3f} V")
print(f"V1 = {V1:.4f}、V2 = {V2:.4f}、Z = {Zid.real:.3f} {Zid.imag:+.1f}j Ω、C = 1/(ω|Im Z|) = {C_back*1e6:.2f} µF")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
tt = np.linspace(0, 2, 801)
ax[0].plot(tt, v1f(tt), label="v1（駆動側）")
ax[0].plot(tt, v2f(tt), label="v2（コンデンサ側）")
ax[0].plot(tt, v2f(tt) * np.cos(w * tt), ":", label="v2·cos ωt")
ax[0].set(xlabel="時間 [s]", ylabel="電圧 [V]", title="(a) 波形と、cos ωt を掛けたもの")
ax[1].plot([0, V2.real], [0, V2.imag], "o-", label="V2")
ax[1].plot([0, V1.real], [0, V1.imag], "s-", label="V1")
ax[1].set(xlabel="実部 [V]", ylabel="虚部 [V]", title="(b) フェーザ", aspect="equal")
for x in ax:
    x.grid(True, alpha=0.3)
    x.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# 確かめ（3-3-3 と図3-6）
ok(near(mc, -0.473, 0.001) and near(ms, 0.753, 0.001), "1周期で平均すると −0.473 と +0.753 が残る（その2倍が V2 の実部と、符号を変えた虚部）")
ok(abs(2 * mc - V2.real) < 1e-12 and abs(-2 * ms - V2.imag) < 1e-12, "V2 = 2·平均(v2 cos ωt) − 2j·平均(v2 sin ωt)")
ok(abs(v2f(tk).mean() - 1.65) < 1e-9, "直流の 1.65V は1周期の平均で消える（cos・sin の平均が0）")
ok(near(Zid.imag, -1592, 1) and abs(Zid.real) < 1 and near(C_back * 1e6, 100.0, 0.1), "Z ≈ −j1592 Ω で、C = 100.0 µF に戻る")

## 4. デジタルロックインでインピーダンスを測る（3-3-3、手順 C の 11・12）

### Pico での記録（読むだけ。このノートブックでは実行しません）

教科書では、Pico B に C SDK の `lockin.c` をビルドして書き込み、Mac の `lockin_capture.py` で10個の周波数（1Hz〜1kHz）の波形を `data_lockin/○○.npz`（`dut`・`rref`・`f`・`n`・`p`・`fs`・`raw0`〜`raw9`）に記録します（`run_ch3.sh` の「C」）。

**Pico B（C SDK）｜lockin/lockin.c**

```c
// 第3章 C：デジタルロックイン用の記録（Pico B / C SDK）
// GP15 から方形波（0 / 3.3V）を出し、R_ref を通してコンデンサにかける。
// ADC0（GP26）= 駆動側（R_ref の手前）、ADC1（GP27）= コンデンサの＋ を、ラウンドロビン＋DMA で等間隔に記録し、
// 生の値（12ビット）を USB シリアルに 16進数で送る。ロックイン（位相の計算）は Mac の lockin_analysis.py で行う。
//
// シリアルコマンド：PING → PONG、RUN → 周波数ごとに「F,周波数,N,P,全体のサンプリング周波数」の行と
//                  16進数のデータ行を出し、最後に END
#include <stdio.h>
#include <string.h>
#include "pico/stdlib.h"
#include "hardware/adc.h"
#include "hardware/dma.h"
#include "hardware/gpio.h"

#define DRIVE_PIN 15                 // 方形波の出力（ピン20）
#define NBUF 40000                   // 記録できるサンプル数（2チャネル合計）
#define SETTLE_MIN_MS 5000           // 周波数を変えたあとの最低の待ち時間 [ms]

static uint16_t buf[NBUF];
static const float FREQS[] = {1, 2, 5, 10, 20, 50, 100, 200, 500, 1000};   // 半周期が整数 µs になる値
#define NF (sizeof(FREQS) / sizeof(FREQS[0]))

static volatile bool level = false;
static int dma_chan;

static bool toggle_cb(repeating_timer_t *t) {
    (void)t;
    level = !level;
    gpio_put(DRIVE_PIN, level);
    return true;                     // くり返す
}

static void measure(float f) {
    int n = 200;                                     // 1周期あたりの点数（各チャネル）
    while (2.0f * n * f < 800.0f) n *= 2;            // ADC の最低の速さ（約730 S/s）を下回らないように
    int p = (f < 2.0f) ? 4 : (f < 10.0f) ? 8 : 20;   // 記録する周期の数
    if (2 * n * p > NBUF) p = NBUF / (2 * n);
    int count = 2 * n * p;

    float fs_total = 2.0f * n * f;                   // 2チャネル合計のサンプリング周波数
    float div = 48000000.0f / fs_total - 1.0f;       // ADC のクロック 48MHz（RP2350 データシートで要確認）
    adc_set_clkdiv(div);
    uint32_t d256 = (uint32_t)(div * 256.0f);        // 実際に設定される値（小数部8ビット）
    float fs_real = 48000000.0f / (1.0f + d256 / 256.0f);

    adc_run(false);
    adc_fifo_drain();
    adc_select_input(0);                             // ADC0 から始めて 0, 1, 0, 1, …
    adc_set_round_robin((1u << 0) | (1u << 1));
    adc_fifo_setup(true, true, 1, false, false);     // FIFO を使い、DMA に知らせる。12ビットのまま

    dma_channel_config c = dma_channel_get_default_config(dma_chan);
    channel_config_set_transfer_data_size(&c, DMA_SIZE_16);
    channel_config_set_read_increment(&c, false);
    channel_config_set_write_increment(&c, true);
    channel_config_set_dreq(&c, DREQ_ADC);
    dma_channel_configure(dma_chan, &c, buf, &adc_hw->fifo, count, true);
    adc_run(true);
    dma_channel_wait_for_finish_blocking(dma_chan);
    adc_run(false);
    adc_fifo_drain();
    adc_set_round_robin(0);

    printf("F,%.3f,%d,%d,%.4f\n", f, n, p, fs_real);
    for (int i = 0; i < count; i++) {
        printf("%03x", buf[i] & 0xfff);
        if (i % 32 == 31 || i == count - 1) printf("\n");
    }
}

int main(void) {
    stdio_init_all();
    gpio_init(DRIVE_PIN);
    gpio_set_dir(DRIVE_PIN, GPIO_OUT);
    gpio_put(DRIVE_PIN, 0);
    adc_init();
    adc_gpio_init(26);                               // デジタル入力を切って ADC 用にする（E9 対策にもなる）
    adc_gpio_init(27);
    dma_chan = dma_claim_unused_channel(true);

    char line[32];
    int len = 0;
    while (true) {
        int ch = getchar_timeout_us(100000);
        if (ch == PICO_ERROR_TIMEOUT) continue;
        if (ch != '\r' && ch != '\n') {
            if (len < 31) line[len++] = (char)ch;
            continue;
        }
        line[len] = 0;
        len = 0;
        if (strcmp(line, "PING") == 0) {
            printf("PONG\n");
        } else if (strcmp(line, "RUN") == 0) {
            for (unsigned k = 0; k < NF; k++) {
                float f = FREQS[k];
                repeating_timer_t timer;
                level = false;
                gpio_put(DRIVE_PIN, 0);
                // 負の値 = 前回の呼び出しの開始から数える（周期がずれない）
                add_repeating_timer_us(-(int64_t)(500000.0f / f), toggle_cb, NULL, &timer);
                int settle = (int)(5000.0f / f);
                sleep_ms(settle > SETTLE_MIN_MS ? settle : SETTLE_MIN_MS);
                measure(f);
                cancel_repeating_timer(&timer);
            }
            gpio_put(DRIVE_PIN, 0);
            printf("END\n");
        } else if (line[0] != 0) {
            printf("ERR unknown: %s\n", line);
        }
    }
}
```

**Pico B（C SDK）｜CMakeLists.txt の変更点**

```cmake
# VS Code の Raspberry Pi Pico 拡張で「New C/C++ Project」（Board: Pico 2 W、名前: lockin）を作り、
# lockin.c を上のコードに置き換えてから、CMakeLists.txt の該当部分を次のようにする
target_link_libraries(lockin
        pico_stdlib
        hardware_adc
        hardware_dma)

# USBシリアルを有効にする（なければ足す）
pico_enable_stdio_usb(lockin 1)
pico_enable_stdio_uart(lockin 0)
```

**Mac｜~/home-lab/ch3/lockin_capture.py**

```python
#!/usr/bin/env python3
"""第3章 C：Pico B（lockin.c）から生の ADC の値を受け取り、npz ファイルに保存する

使い方: cd ~/home-lab/ch3 && uv run python lockin_capture.py --dut C100u --rref 998 --out data_lockin/C100u_1.npz
  --dut  : 測るもの（C100u、C0u1、校正用の抵抗 R1k など）
  --rref : R_ref の実測値 [Ω]
"""
import argparse
import glob
import pathlib
import sys
import time

import numpy as np
import serial


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dut", required=True)
    ap.add_argument("--rref", type=float, required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--port", default=None)
    a = ap.parse_args()
    out = pathlib.Path(a.out)
    if out.exists():
        sys.exit(f"{out} はすでにあります（上書きしないように止めました）")
    out.parent.mkdir(parents=True, exist_ok=True)
    ports = sorted(glob.glob("/dev/cu.usbmodem*"))
    if not ports and not a.port:
        sys.exit("Pico B のシリアルポートが見つかりません")
    s = serial.Serial(a.port or ports[0], 115200, timeout=2)
    time.sleep(0.3)
    s.reset_input_buffer()
    s.write(b"PING\n")
    if s.readline().decode(errors="ignore").strip() != "PONG":
        sys.exit("PONG が返りません。Pico B に lockin の UF2 が書き込まれているか確認してください")
    print("測定を始めます（周波数10個。全部で数分かかる）")
    s.write(b"RUN\n")
    data, cur, hexbuf = {}, None, []
    t_last = time.time()
    while True:
        ln = s.readline().decode(errors="ignore").strip()
        if not ln:
            if time.time() - t_last > 120:
                sys.exit("2分以上データが来ません")
            continue
        t_last = time.time()
        if ln.startswith(("F,", "END")):
            if cur is not None:
                h = "".join(hexbuf)
                raw = np.array([int(h[i:i + 3], 16) for i in range(0, len(h), 3)], dtype=np.uint16)
                data[cur] = raw
                print(f"  {cur[0]:7.1f} Hz：{len(raw)} サンプル")
            if ln == "END":
                break
            _, f, n, p, fs = ln.split(",")
            cur, hexbuf = (float(f), int(n), int(p), float(fs)), []
        elif ln.startswith("ERR"):
            sys.exit(ln)
        else:
            hexbuf.append(ln)
    s.close()
    keys = sorted(data)
    np.savez(out, dut=a.dut, rref=a.rref,
             f=np.array([k[0] for k in keys]), n=np.array([k[1] for k in keys]),
             p=np.array([k[2] for k in keys]), fs=np.array([k[3] for k in keys]),
             **{f"raw{i}": data[k] for i, k in enumerate(keys)})
    print(f"{out} に保存しました")


if __name__ == "__main__":
    main()
```

### 合成データで解析する

`synth_lockin.py` は、`lockin.c` と同じ周波数・点数・周期の数で、方形波で駆動した $Z = R_s + 1/(j\omega C)$ の定常応答を ADC0・ADC1 の交互の標本として作ります（開始の位相はでたらめ、雑音 0.6mV、12ビット）。まず 100µF（$R_s = 0.3\,\Omega$）・$R_{\mathrm{ref}} = 998\,\Omega$ です（`run_ch3.sh` の「C」の2行目と同じ引数で解析します）。

In [ ]:
run("synth_lockin.py 100e-6 0.3 998 data_lockin/C100u_1.npz")
out_c100 = run("lockin_analysis.py data_lockin/C100u_1.npz --rref-rel 0.005")
show("data_lockin/C100u_1.png")

In [ ]:
# 確かめ（手順 C の 11 と 3-7 の C）
import lockin_analysis as LA


def fit_line(text, name):
    m = re.search(rf"{name}\s+= ([-\d.]+) (?:µF|Ω)（95%区間 ([-\d.]+)〜([-\d.]+)", text)
    return tuple(map(float, m.groups()))


Cl = fit_line(out_c100, "C")
Rs = fit_line(out_c100, "R_s")
ok(abs(Cl[0] / 100 - 1) < 0.02 and Cl[1] < 100 < Cl[2], f"C = {Cl[0]} µF（真の値 100 µF を区間 {Cl[1]}〜{Cl[2]} に含む。数%以内）")
ok(Rs[1] < 0.3 < Rs[2], f"R_s = {Rs[0]} Ω（95%区間 {Rs[1]}〜{Rs[2]} に真の値 0.3 Ω を含む）")
res, _, _ = LA.impedance("data_lockin/C100u_1.npz")
fr = np.array([r["f"] for r in res])
slope = np.polyfit(np.log10(fr), np.log10(-np.array([r["Z"].imag for r in res])), 1)[0]
ok(abs(slope + 1) < 0.02, f"−Im Z は周波数に反比例して減る（ボード線図の傾き {slope:.3f} ≈ −1）")
ok(num(r"規格化した残差の2乗和/自由度 = ([\d.]+)", out_c100) < 3, "R_s + C のモデルで合う（規格化した残差の2乗和/自由度が 3 より小さい）")

次に 0.1µF（$R_s = 0$）を $R_{\mathrm{ref}} = 100$kΩ で測ったつもりのデータです（`run_ch3.sh` の「C」の残りの3行）。低い周波数では $|Z|/R_{\mathrm{ref}}$ が 1 より大きく（1Hz で約16）、表の「系統」の列がとても大きくなります。`--fmin 20` で 20Hz より低い点を外しても C がほぼ同じなら安心できます。

In [ ]:
run("synth_lockin.py 0.1e-6 0 99800 data_lockin/C0u1_1.npz")
out_c01 = run("lockin_analysis.py data_lockin/C0u1_1.npz --rref-rel 0.005")
out_c01f = run("lockin_analysis.py data_lockin/C0u1_1.npz --rref-rel 0.005 --fmin 20")

In [ ]:
# 確かめ（手順 C の 12）
C1, C2 = fit_line(out_c01, "C"), fit_line(out_c01f, "C")
ok(abs(C1[0] / 0.1 - 1) < 0.02 and C1[1] < 0.1 < C1[2], f"0.1µF：C = {C1[0]} µF（区間 {C1[1]}〜{C1[2]} が真の値を含む）")
ok(abs(C2[0] / C1[0] - 1) < 0.01, f"--fmin 20 で低い周波数を外しても C がほぼ同じ（{C2[0]} µF、差 1% 未満）")
ok(out_c01f.count("（当てはめに使わない）") == 4, "--fmin 20 では 1・2・5・10Hz の4点が当てはめから外れた")
rows = re.findall(r"^\s+([\d.]+)\s+[-\d.]+ ±\s+[\d.]+ ±\s+([\d.e+]+)\s+[\d.]+ ±\s+[\d.]+\s+[-\d.]+\s+([\d.]+)", out_c01, re.M)
sysd = {float(fq): (float(e), float(r)) for fq, e, r in rows}
ok(len(sysd) == 10 and sysd[1.0][1] > 10 and sysd[1.0][0] > 100 * sysd[1000.0][0],
   f"1Hz では |Z|/R_ref = {sysd[1.0][1]}（本文の目安は約16）で、系統誤差 {sysd[1.0][0]:.3g} Ω は 1kHz の {sysd[1000.0][0]:.3g} Ω の100倍より大きい")
ok("R_s の区間が 0 をまたぐ" in out_c01, "0.1µF では R_s（ESR）は決められない（区間が 0 をまたぐ）")

最後に、3-3-3 の「精度の限界」の段落の例（0.1µF を $R_{\mathrm{ref}} = 10$kΩ で測ったとき）を確かめます。`lockin_analysis.py` の系統誤差の重み $|Z|\,|R_{\mathrm{ref}} + Z|/R_{\mathrm{ref}}\cdot(\pi/N)/\sqrt3$ を、$|Z|(\pi/N)/\sqrt3$ に替えて当てはめ直すと、$|Z| > R_{\mathrm{ref}}$ の点を信じすぎて C が大きくずれます。本文の値（$0.130\,\mu$F と $0.1004\,\mu$F）は別の合成データのものなので、ここでの数値は少し違います。

In [ ]:
run("synth_lockin.py 0.1e-6 0 10000 data_lockin/C0u1_10k.npz")
from scipy.optimize import least_squares
res, _, rref = LA.impedance("data_lockin/C0u1_10k.npz")
fr = np.array([r["f"] for r in res]); Zm = np.array([r["Z"] for r in res])
se = np.array([r["se"] for r in res]); nn = np.array([r["n"] for r in res]); wr = 2 * np.pi * fr
Cfit = {}
for name, sy in [("|Z|(π/N)/√3（入れないと）", np.abs(Zm) * (np.pi / nn) / np.sqrt(3)),
                 ("|Z||R_ref+Z|/R_ref·(π/N)/√3（lockin_analysis.py）", np.abs(Zm) * np.abs(rref + Zm) / rref * (np.pi / nn) / np.sqrt(3))]:
    sre, sim = np.sqrt(se.real ** 2 + sy ** 2), np.sqrt(se.imag ** 2 + sy ** 2)

    def resid(th):
        zm = th[0] + 1 / (1j * wr * np.exp(th[1]))
        return np.concatenate([(zm.real - Zm.real) / sre, (zm.imag - Zm.imag) / sim])

    x = least_squares(resid, [0.5, np.log(np.median(-1 / (wr * Zm.imag)))]).x
    Cfit[name] = np.exp(x[1]) * 1e6
    print(f"重み {name}：C = {Cfit[name]:.4f} µF")
out_c10k = run("lockin_analysis.py data_lockin/C0u1_10k.npz --rref-rel 0.005")

In [ ]:
# 確かめ（3-3-3 の「方形波の標本化による位相の誤差」）
wrong, right = Cfit.values()
ok(abs(wrong / 0.1 - 1) > 0.1 and abs(right / 0.1 - 1) < 0.01,
   f"|Z|(π/N)/√3 の重みでは C が {wrong:.3f} µF と10%以上ずれ、本文の式の重みでは {right:.4f} µF（1% 以内）")
ok(abs(fit_line(out_c10k, "C")[0] - right) < 1e-3, "lockin_analysis.py の表示の C と同じ")

## おわりに

- RC の合成データで、毎回同じように出る遅い成分（誘電吸収）は、1回の当てはめの区間にも回ごとのばらつきにも現れず、当てはめる範囲を変えると C が動くことで初めて分かる偏りになることを確かめました。
- 漏れ電流では、温度だけで回帰すると符号まで逆になり（見かけの回帰）、状態空間モデルでも水準の動き方の選び方で β が大きく変わることを見ました。
- ロックインでは、整数周期の平均で基本波だけが残ること、系統誤差（方形波のエッジの位置）を重みに入れないと $|Z| > R_{\mathrm{ref}}$ の点で C までずれることを確かめました。
- このノートブックに入れていないもの：Pico A・Pico B での記録、Pi のロガー、C SDK のビルド、テスタでの測定。これらは Mac・Pico・Pi で教科書の手順のとおりに行います。自分の記録（`data_rc/` の CSV・JSON、`lab_snapshot.db`、`data_lockin/` の npz）を Colab の左のファイルの欄に上げれば、同じ `run(...)` で解析できるはずです（執筆時には確かめていません）。